# Old34 team-review narrative

Experiment: `20260525_060508_stage0_mrd_old34_broad_screen_82db5093`  
Companion guide: `old34_inquiry_narrative_guide.md`

This notebook is a **curated review layer** over completed artifacts. It performs no Stage 1 or Stage 2 fitting. Use it to prepare a PI/team presentation, then open the source notebooks for implementation details.

## Source-notebook sequence

1. `stage2_old34_comprehensive_diagnostic_suite.ipynb` — D1–D3, 643 representations → 41-panel diagnosis.
2. `stage2_old34_layer4_5_interpretability_suite.ipynb` — D4–D5, regularization/coefficient pathology and factor→gene grounding.
3. `stage2_old34_meeting_followup_metric_interactions.ipynb` — D6–D9, AP baseline, patient support, interactions, IFN-α attribution, and 41-panel direct-gene baselines.
4. `stage2_old34_donor_downsampling_sensitivity.ipynb` — D10–D11, donor abundance, coefficient collapse, and corrected factor stability.

The Jun-4 Figure-3 notebook and July dossier are historical inputs. They explain why these diagnostics were built, but their implementation-status statements are now stale.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "sc_classification").exists():
            return candidate
    raise FileNotFoundError("Could not locate sc_classification repository root")


REPO_ROOT = find_repo_root()
EXPERIMENT_ID = "20260525_060508_stage0_mrd_old34_broad_screen_82db5093"
EXP_DIR = REPO_ROOT / "experiments" / EXPERIMENT_ID
DIAGNOSTIC_DIR = EXP_DIR / "analysis" / "scorecards" / "stage2_diagnostic"
MEETING_TABLE_DIR = DIAGNOSTIC_DIR / "meeting_followup_v1"
DONOR_TABLE_DIR = DIAGNOSTIC_DIR / "donor_sensitivity_v1"
MEETING_FIGURE_DIR = EXP_DIR / "analysis" / "figures" / "stage2_diagnostic" / "meeting_followup_v1"
DONOR_FIGURE_DIR = EXP_DIR / "analysis" / "figures" / "stage2_diagnostic" / "donor_sensitivity_v1"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 60)
print("Experiment:", EXP_DIR)
print("This notebook reads completed CSV/PNG artifacts only.")

Experiment: /home/minhang/mds_project/sc_classification/experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093
This notebook reads completed CSV/PNG artifacts only.


## 1. Metric contract: AP, prevalence, and one operating threshold

For a held-out patient with malignant prevalence

\[
\pi=\frac{N_+}{N_++N_-},
\]

scikit-learn average precision is

\[
\operatorname{AP}=\sum_n(R_n-R_{n-1})P_n.
\]

If every cell receives the same score, all cells enter the predicted-positive set in one tied step. Recall increases from 0 to 1 and precision at that step is \(\pi\), so

\[
\operatorname{AP}_{\mathrm{constant}}=(1-0)\pi=\pi.
\]

Thus the gray baseline is properly labeled **constant-score no-skill AP (= prevalence)**. An all-healthy and all-malignant constant score have the same AP ranking baseline.

AUPRC/AP still sweeps score thresholds, like AUROC. “Threshold utility” means choosing one particular operating cutoff \(t\), producing \(\hat y_i=\mathbf 1[s_i\ge t]\), and evaluating accuracy, sensitivity, specificity, or missed-cancer cost. AP does not select that operating cutoff or encode those costs.

Lift and ceiling-normalized lift are

\[
L_h=\operatorname{AP}_h-\pi_h,
\qquad
L_h^{\mathrm{norm}}=\frac{\operatorname{AP}_h-\pi_h}{1-\pi_h}.
\]

### Spearman association

For each representation \(r\), rank its seven held-out metric values and the seven patient prevalence values:

\[
\rho_r=\operatorname{corr}\{\operatorname{rank}(M_{r,h}),\operatorname{rank}(\pi_h)\}.
\]

Repeat for 643 representations, then report \(\operatorname{median}_r\rho_r\). This is **rho (ρ), not a p-value**. Nominal SciPy p-values are stored but are not inferential with seven patients and 643 dependent representations.

Verified medians: AP–prevalence ρ = 0.964; raw-lift–prevalence ρ = 0.821; normalized-lift–prevalence ρ = 0.821. The association means patient support and biological separability are not identified separately in this cohort; it does not prove that the signal is non-biological.

In [2]:
composition = pd.read_csv(MEETING_TABLE_DIR / "mrd_patient_cell_composition.csv")
dummy = pd.read_csv(MEETING_TABLE_DIR / "mrd_dummy_policy_metrics.csv")
association = pd.read_csv(MEETING_TABLE_DIR / "old34_metric_prevalence_association_summary.csv")

print("Exact patient composition")
display(
    composition[[
        "patient_id",
        "n_malignant_recipient",
        "n_healthy_donor",
        "n_healthy_recipient",
        "n_total_binary_cells",
        "malignant_prevalence",
        "supported_for_lift_summary",
    ]].style.format({"malignant_prevalence": "{:.3%}"})
)

print("Median hard-policy utility across the seven supported patients")
display(
    dummy.loc[dummy["patient_id"].isin(["P01", "P02", "P03", "P04", "P06", "P07", "P09"])]
    .groupby("policy")[[
        "constant_score_average_precision",
        "accuracy",
        "malignant_sensitivity",
        "healthy_specificity",
        "missed_malignant_cells",
    ]]
    .median()
)

print("Distribution of within-representation Spearman rho values")
display(
    association.loc[
        association["covariate"].isin(["patient_prevalence", "n_malignant_h"]),
        ["metric", "covariate", "median_spearman_rho", "p20_spearman_rho", "p80_spearman_rho", "n_representations"],
    ]
)

Exact patient composition


,patient_id,n_malignant_recipient,n_healthy_donor,n_healthy_recipient,n_total_binary_cells,malignant_prevalence,supported_for_lift_summary
0,P01,45,11815,0,11860,0.379%,True
1,P02,211,1234,17,1462,14.432%,True
2,P03,758,1568,1,2327,32.574%,True
3,P04,102,1012,14,1128,9.043%,True
4,P05,8,4341,10,4359,0.184%,False
5,P06,44,3214,1,3259,1.350%,True
6,P07,22,2163,0,2185,1.007%,True
7,P08,0,1411,0,1411,0.000%,False
8,P09,375,16745,6,17126,2.190%,True
9,P10,0,2715,0,2715,0.000%,False


Median hard-policy utility across the seven supported patients


,constant_score_average_precision,accuracy,malignant_sensitivity,healthy_specificity,missed_malignant_cells
policy,,,,,
all_healthy_majority_policy,0.021897,0.978103,0.0,1.0,102.0
all_malignant_sensitivity_policy,0.021897,0.021897,1.0,0.0,0.0


Distribution of within-representation Spearman rho values


,metric,covariate,median_spearman_rho,p20_spearman_rho,p80_spearman_rho,n_representations
0,heldout_auprc,n_malignant_h,0.750000,0.607143,0.785714,643
2,heldout_auprc,patient_prevalence,0.964286,0.892857,1.000000,643
3,heldout_auroc,n_malignant_h,0.571429,0.285714,0.785714,643
5,heldout_auroc,patient_prevalence,0.500000,0.107143,0.714286,643
6,heldout_lift,n_malignant_h,0.607143,0.300000,0.785714,643
8,heldout_lift,patient_prevalence,0.821429,0.321429,0.857143,643
9,normalized_heldout_lift,n_malignant_h,0.678571,0.535714,0.821429,643
11,normalized_heldout_lift,patient_prevalence,0.821429,0.683403,0.892857,643


### Presentation plots for metric validity and cohort support

**D6A — exact malignant/donor-healthy/recipient-healthy composition by patient**

![D6A patient composition](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/meeting_followup_v1/D6A_mrd_patient_cell_composition.png)

**D6D — distribution across 643 within-representation Spearman correlations**

![D6D metric association](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/meeting_followup_v1/D6D_metric_association_with_prevalence_and_counts.png)

**D6E — same constant-score AP, opposite hard-decision utility**

![D6E policy utility](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/meeting_followup_v1/D6E_dummy_policy_utility_comparison.png)

## 2. No-DR direct-gene baseline

The no-DR comparator uses normalized/log1p panel expression and fits regularized logistic regression directly on standardized gene columns:

\[
\Pr(y_i=1\mid\mathbf x_i)=\sigma\left(\beta_0+\sum_{g=1}^{G}\beta_g z_{ig}\right).
\]

It therefore maps named genes—not latent factors—to the malignancy score. Discovery and LOPO paths are complete for 41/41 biological panels.

For each panel, the exploratory comparison selects the DR method/K with the largest pooled-cell LOPO AP and compares it with the one direct-gene representation:

- best DR wins pooled AP in 38/41 panels;
- best DR wins seven-patient median lift in 16/41;
- best DR wins seven-patient P20 lift in 17/41.

This is not contradictory. Pooled AP is cell/support weighted, whereas median and P20 give patients equal influence. “Best DR” is also selected from many candidates on the reported pooled criterion, so 38/41 is optimistic until nested or external validation.

In [3]:
direct_vs_dr = pd.read_csv(MEETING_TABLE_DIR / "old34_direct_gene_vs_best_dr_summary.csv")
priority_ids = [
    "single_group__stress_arrest",
    "single_geneset__hallmark_p53_pathway",
    "single_geneset__hallmark_interferon_alpha_response",
]

win_counts = {
    "pooled_lopo_ap": int(direct_vs_dr["dr_value_added_pooled_lopo_ap"].gt(0).sum()),
    "median_lift": int(direct_vs_dr["dr_value_added_median_lift"].gt(0).sum()),
    "p20_lift": int(direct_vs_dr["dr_value_added_p20_lift"].gt(0).sum()),
}
print("Strict best-DR win counts out of 41 panels:", win_counts)
print("Priority-panel comparisons")
display(
    direct_vs_dr.loc[
        direct_vs_dr["stage0_panel_id"].isin(priority_ids),
        [
            "panel_label_with_genes",
            "best_dr_method",
            "best_dr_k",
            "direct_pooled_lopo_ap",
            "best_dr_pooled_lopo_ap",
            "direct_median_lift",
            "best_dr_median_lift",
            "direct_p20_lift",
            "best_dr_p20_lift",
        ],
    ]
)

Strict best-DR win counts out of 41 panels: {'pooled_lopo_ap': 38, 'median_lift': 16, 'p20_lift': 17}
Priority-panel comparisons


,panel_label_with_genes,best_dr_method,best_dr_k,direct_pooled_lopo_ap,best_dr_pooled_lopo_ap,direct_median_lift,best_dr_median_lift,direct_p20_lift,best_dr_p20_lift
12,interferon alpha response (92),fa,10,0.048606,0.200924,0.012658,-2.949030e-17,0.003720,-4.649059e-17
18,p53 pathway (164),fa,10,0.070474,0.175180,0.128392,-2.949030e-17,0.023085,-4.649059e-17
39,stress arrest (685),factosig,40,0.110621,0.209094,0.186741,1.106517e-01,0.040794,3.972980e-02


### Presentation plots for Stage 1 value added

**D9A — pooled-cell LOPO AP: direct genes versus selected best DR**

![D9A direct versus DR pooled AP](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/meeting_followup_v1/D9A_direct_gene_vs_best_dr_pooled_lopo_ap.png)

**D9B — seven-patient lower-tail P20 lift**

![D9B direct versus DR P20](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/meeting_followup_v1/D9B_direct_gene_vs_best_dr_p20_lift.png)

**D9C — patient-specific best-DR minus direct-gene lift**

![D9C patient value added](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/meeting_followup_v1/D9C_best_dr_minus_direct_gene_patient_lift.png)

## 3. IFN-α PCA40 versus FA40 attribution

The comparison fixes the 92-gene panel, K=40, seed 42, and globally selected L1 \(C=10^{-4}\).

For held-out patient \(h\), the stored factor coefficients reconstruct the ranking score:

\[
s_{ih}=\sum_{j=1}^{40}\beta_{hj}z_{ihj}.
\]

For fixed-model ablation of factor \(j\):

\[
\Delta\operatorname{AP}_{h,j}=\operatorname{AP}(s_h)-\operatorname{AP}(s_h-\beta_{hj}z_{hj}).
\]

Reconstructed AP matches stored AP within \(1.4\times10^{-6}\).

Observed fold evidence:

- PCA factor 008 is the only active PCA factor in P02, P03, P04, P06, and P07.
- FA factor 007 is active only in P03 and P04.
- FA has an all-zero selected coefficient vector in P02 and P07.
- PCA factor-008 scores correlate most with FA factor 007 (−0.685) and factor 008 (+0.580).

The precise conclusion is: **PCA retains factor 008 in P02/P07 while FA’s sparse selected model retains no factor there.** PCA factor-008 variation is split mainly across FA axes 007/008. This is a geometry × regularization interaction, not proof that FA lacks the underlying biology.

In [4]:
ifna_active = pd.read_csv(MEETING_TABLE_DIR / "interferon_alpha_pca40_fa40_active_factors_by_fold.csv")
fa_correlations = pd.read_csv(MEETING_TABLE_DIR / "interferon_alpha_fa_factors_correlated_with_pca_driver.csv")
ifna_reconstruction = pd.read_csv(MEETING_TABLE_DIR / "interferon_alpha_pca40_fa40_ap_reconstruction.csv")

print("Active factor coefficients and fixed-model attribution")
display(
    ifna_active[[
        "stage1_method",
        "heldout_patient",
        "feature_id",
        "coefficient",
        "full_model_ap",
        "ap_drop_when_ablated",
        "single_factor_lift",
    ]]
)
print("FA axes most correlated with PCA factor 008")
display(fa_correlations)
print("Maximum AP reconstruction error:", ifna_reconstruction["absolute_reconstruction_error"].max())

Active factor coefficients and fixed-model attribution


,stage1_method,heldout_patient,feature_id,coefficient,full_model_ap,ap_drop_when_ablated,single_factor_lift
0,pca,P02,factor_008,-0.052223,0.303741,0.159419,0.159419
1,pca,P03,factor_008,-0.063708,0.459715,0.133974,0.133974
2,pca,P04,factor_008,-0.039978,0.253098,0.162672,0.162672
3,pca,P06,factor_008,-0.046512,0.014122,0.000621,0.000621
4,pca,P07,factor_008,-0.044955,0.120820,0.110751,0.110751
5,fa,P03,factor_007,0.009582,0.469400,0.143659,0.143659
6,fa,P04,factor_007,0.003695,0.289146,0.198720,0.198720


FA axes most correlated with PCA factor 008


,fa_feature_id,score_correlation_to_pca_driver,absolute_score_correlation_to_pca_driver
0,factor_007,-0.684621,0.684621
1,factor_008,0.580002,0.580002


Maximum AP reconstruction error: 1.3845020665481655e-06


### Presentation plots for the IFN-α case study

**D7D — patient-specific PCA40 versus FA40 lift**

![D7D IFN patient comparison](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/meeting_followup_v1/D7D_interferon_alpha_pca40_vs_fa40.png)

**D8B — active factor coefficients by held-out fold**

![D8B active IFN factors](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/meeting_followup_v1/D8B_interferon_alpha_active_factor_coefficients.png)

**D8C — full score versus driver ablation versus no-skill AP**

![D8C IFN factor ablation](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/meeting_followup_v1/D8C_interferon_alpha_driver_factor_ablation.png)

**D8D/E — gene loadings and PCA-factor correlation across FA axes**

![D8D IFN loading genes](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/meeting_followup_v1/D8D_interferon_alpha_driver_loading_genes.png)

![D8E PCA driver correlations](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/meeting_followup_v1/D8E_interferon_alpha_pca_driver_correlations_to_fa.png)

## 4. Donor downsampling: separate Stage 2 collapse from Stage 1 movement

Donor-healthy training cells were retained at fractions 1.0, 0.5, 0.25, and 0.1, stratified by patient × predicted cell type. All malignant and recipient-healthy cells were preserved, and each full held-out patient remained unchanged.

At every downsampled fraction, the fixed L1 \(C=10^{-4}\) IFN-α models cross to

\[
\boldsymbol\beta=\mathbf 0.
\]

Their held-out score becomes constant and AP returns to prevalence. This is an **all-zero sparse-regularization collapse**. It does not show that a retained nonzero classifier learned or lost a donor-origin shortcut.

Stage 1 basis movement is measured independently after one-to-one factor alignment:

\[
\text{score stability}=|\operatorname{corr}(\mathbf z_j^{\rm full},\mathbf z_{k(j)}^{\rm down})|,
\]

\[
\text{loading cosine}=\frac{\boldsymbol\lambda^\top\tilde{\boldsymbol\lambda}}{\|\boldsymbol\lambda\|_2\|\tilde{\boldsymbol\lambda}\|_2},
\]

\[
J(A,B)=\frac{|A\cap B|}{|A\cup B|},
\]

where \(A,B\) are the top-10 absolute-loading genes. Jaccard 0.818 means 9/10 overlap; 0.333 means 5/10 overlap.

### Corrected 25% result

The previous ≈0.99/0.818 blanket statement was caused by including four zero-coefficient factors in a “top-five” median. After retaining only factors genuinely supervised in the full-donor model:

- FA factor 007: score correlation 0.952, loading cosine 0.943, Jaccard 0.818.
- PCA factor 008: score correlation 0.854, loading cosine 0.854, Jaccard 0.333.

Thus FA007 remains fairly stable at 25%; PCA008 changes materially at 25% but is strongly stable at 50%. A performance-level donor-shortcut test still needs condition-specific regularization selection or scaling.

In [5]:
collapse = pd.read_csv(DONOR_TABLE_DIR / "fixed_basis_regularization_collapse_summary.csv")
factor_stability = pd.read_csv(DONOR_TABLE_DIR / "ifna_basis_refit_factor_stability_summary.csv")
nonzero_factors = pd.read_csv(DONOR_TABLE_DIR / "ifna_full_donor_nonzero_supervised_factors.csv")

print("Full-donor nonzero supervised factor identities")
display(nonzero_factors)
print("All-zero Stage 2 fold frequency at 25% donor retention")
display(
    collapse.loc[
        collapse["donor_fraction"].eq(0.25),
        ["target_label", "zero_solution_fraction", "median_active_coefficients"],
    ].sort_values("target_label")
)
print("Corrected factor-specific Stage 1 stability at 25% donor retention")
display(
    factor_stability.loc[
        factor_stability["donor_fraction"].eq(0.25) & factor_stability["is_driver_factor"],
        [
            "target_label",
            "median_score_correlation",
            "median_loading_cosine",
            "median_top10_gene_jaccard",
            "n_reference_factors",
            "n_factor_seed_rows",
        ],
    ]
)

Full-donor nonzero supervised factor identities


,target_id,feature_id,mean_absolute_coefficient,is_driver_factor
0,ifna_fa40,factor_007,0.001882,True
1,ifna_pca40,factor_008,0.035203,True


All-zero Stage 2 fold frequency at 25% donor retention


,target_label,zero_solution_fraction,median_active_coefficients
5,IFN-α | FA40,1.0,0.0
9,IFN-α | PCA40,1.0,0.0
1,IFN-α | direct genes,1.0,0.0
21,Stress arrest | best DR,0.0,16.0
25,Stress arrest | direct genes,0.0,685.0
13,p53 | best DR,1.0,0.0
17,p53 | direct genes,0.0,164.0


Corrected factor-specific Stage 1 stability at 25% donor retention


,target_label,median_score_correlation,median_loading_cosine,median_top10_gene_jaccard,n_reference_factors,n_factor_seed_rows
3,IFN-α | FA40,0.951913,0.943101,0.818182,1,3
11,IFN-α | PCA40,0.854429,0.853678,0.333333,1,3


### Presentation plots for donor sensitivity

**D10C — patient-specific AP change at 25% donor retention**

![D10C patient donor sensitivity](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/donor_sensitivity_v1/D10C_fixed_basis_patient_ap_change_25pct.png)

**D10E — show this before attributing AP loss to donor biology**

![D10E all-zero coefficient collapse](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/donor_sensitivity_v1/D10E_fixed_basis_zero_coefficient_collapse.png)

**D11B — aligned score stability, with the genuine supervised factor separated**

![D11B factor score stability](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/donor_sensitivity_v1/D11B_ifna_refit_factor_score_stability.png)

**D11C — corrected nonzero-supervised-factor loading and top-gene stability**

![D11C corrected loading stability](../../../../experiments/20260525_060508_stage0_mrd_old34_broad_screen_82db5093/analysis/figures/stage2_diagnostic/donor_sensitivity_v1/D11C_ifna_refit_driver_loading_stability.png)

## 5. Proposed team narrative and claim gate

### Suggested presentation order

1. Stage 0 panel → Stage 1 latent/direct representation → Stage 2 probe.
2. D6A cohort composition and seven-patient support boundary.
3. Constant-score AP proof and D6E threshold-utility distinction.
4. D6D prevalence/support association: state non-identifiability, not “non-biological.”
5. D1/D2 or D7A/D7B panel × method × patient landscape.
6. D9 direct-gene baseline: broad pooled DR gain, mixed patient-equal gain.
7. D7D + D8B/C/E IFN-α case study.
8. D10E before D11B/C donor sensitivity and corrected factor result.
9. End with the larger-cohort/statistical-control plan.

### Descriptive findings currently supported

- There is strong patient × panel × DR-method/K heterogeneity.
- Constant tied-score AP equals patient prevalence exactly.
- Lift remains positively associated with prevalence/support across representations.
- No-DR gene-level baselines are complete for 41/41 panels.
- DR usually improves pooled AP but not consistently median/P20 lift.
- PCA008 is retained in P02/P07 while the selected FA sparse model is all-zero.
- Donor downsampling exposes a sparse-regularization collapse.
- At 25%, FA007 is more stable than PCA008; both are strongly stable at 50%.

### Not yet publication-level claims

- High lift is caused by patient biology rather than support.
- FA is intrinsically inferior to PCA for IFN-α.
- The classifier is invariant to donor-versus-recipient origin.
- DR universally adds value over direct genes.
- The selected panels beat matched-size random gene sets.
- “Best” configurations are unbiased deployment estimates.

Priority additions with the larger cohort: patient-level uncertainty, matched-size controls, prespecified/nested selection, inductive Stage 1 sensitivity, and donor-downsampling models with condition-specific regularization selection.

In [6]:
expected_figures = [
    MEETING_FIGURE_DIR / "D6A_mrd_patient_cell_composition.png",
    MEETING_FIGURE_DIR / "D6D_metric_association_with_prevalence_and_counts.png",
    MEETING_FIGURE_DIR / "D6E_dummy_policy_utility_comparison.png",
    MEETING_FIGURE_DIR / "D9A_direct_gene_vs_best_dr_pooled_lopo_ap.png",
    MEETING_FIGURE_DIR / "D9B_direct_gene_vs_best_dr_p20_lift.png",
    MEETING_FIGURE_DIR / "D9C_best_dr_minus_direct_gene_patient_lift.png",
    MEETING_FIGURE_DIR / "D7D_interferon_alpha_pca40_vs_fa40.png",
    MEETING_FIGURE_DIR / "D8B_interferon_alpha_active_factor_coefficients.png",
    MEETING_FIGURE_DIR / "D8C_interferon_alpha_driver_factor_ablation.png",
    MEETING_FIGURE_DIR / "D8D_interferon_alpha_driver_loading_genes.png",
    MEETING_FIGURE_DIR / "D8E_interferon_alpha_pca_driver_correlations_to_fa.png",
    DONOR_FIGURE_DIR / "D10C_fixed_basis_patient_ap_change_25pct.png",
    DONOR_FIGURE_DIR / "D10E_fixed_basis_zero_coefficient_collapse.png",
    DONOR_FIGURE_DIR / "D11B_ifna_refit_factor_score_stability.png",
    DONOR_FIGURE_DIR / "D11C_ifna_refit_driver_loading_stability.png",
]
missing_figures = [str(path) for path in expected_figures if not path.exists()]
assert not missing_figures, missing_figures
assert len(composition) == 13
assert len(direct_vs_dr) == 41
assert set(nonzero_factors["feature_id"]) == {"factor_007", "factor_008"}
print(f"Review validation passed: {len(expected_figures)} figures, 13 patients, 41 direct-gene comparisons.")

Review validation passed: 15 figures, 13 patients, 41 direct-gene comparisons.
